# USDA Cropland Data Layer (CropScape) EDA

This notebook:
1. **Loads** USDA Cropland Data Layer (CDL) data at the **county** level via the CropScape REST API.
2. **Explores** crop coverage and land use patterns by county.
3. Runs **basic exploratory data analysis** (EDA) in Python.

* The Cropland Data Layer is a raster, geo-referenced, crop-specific land cover data layer created annually by the USDA NASS. [LINK](https://nassgeodata.gmu.edu/CropScape/)

* Available time range: 2008–present (CONUS)

* The CropScape web service provides `GetCDLStat` for county-level summary statistics (acreage by crop category).

* Target years for this project: 2016–2019 (overlap with pesticide and health data).

* Requires: Python with `pandas`, `numpy`, `matplotlib`, `seaborn`, and `requests`.

In [ ]:
# Load relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO

## 1. Load county-level CDL data via CropScape API

The CropScape `GetCDLStat` service returns acreage by crop category for a given county (FIPS) and year. We sample several counties across states for EDA. For full analysis, you would loop over all counties.

In [ ]:
# CropScape GetCDLStat API - returns XML with a returnURL to the actual data file
CDL_API = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

def get_county_cdl(year: int, fips: str, format: str = "csv") -> pd.DataFrame:
    """Fetch CDL statistics for a county (5-digit FIPS) and year."""
    import xml.etree.ElementTree as ET
    params = {"year": year, "fips": fips, "format": format}
    r = requests.get(CDL_API, params=params)
    r.raise_for_status()
    # Parse XML to get returnURL (API returns URL to data, not data directly)
    root = ET.fromstring(r.text)
    ns = {"ns": "http://cropscape.csiss.gmu.edu/CDLService/"}
    url_elem = root.find(".//ns:returnURL", ns) or root.find(".//returnURL")
    if url_elem is None:
        data_url = f"https://nassgeodata.gmu.edu/webservice/nass_data_cache/byfips/CDL_{year}_{fips}.{format}"
    else:
        data_url = url_elem.text.strip()
    r2 = requests.get(data_url)
    r2.raise_for_status()
    if format == "csv":
        return pd.read_csv(StringIO(r2.text))
    return pd.read_csv(StringIO(r2.text), sep="\t")

# Sample counties for EDA (state FIPS + county FIPS = 5-digit)
# e.g., 06037 = Los Angeles CA, 48201 = Harris TX, 17031 = Cook IL, 36061 = New York NY
SAMPLE_FIPS = ["06037", "48201", "17031", "36061", "41051", "06075", "06067"]  # mix of urban/ag
YEARS = [2016, 2017, 2018, 2019]

dfs = []
for year in YEARS:
    for fips in SAMPLE_FIPS:
        try:
            df = get_county_cdl(year, fips)
            df["YEAR"] = year
            df["FIPS"] = fips
            dfs.append(df)
        except Exception as e:
            print(f"Failed {year} {fips}: {e}")

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
df.head(15)

In [ ]:
# Inspect columns and structure
print("Columns:", list(df.columns))
print("\nShape:", df.shape)
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

## 2. Visualizations

Explore crop acreage distributions and top crops by county.

In [ ]:
# Aggregate acreage by category across all sampled counties/years (if category column exists)
# CDL returns columns like 'Category', 'Acres', 'Value' - names may vary by API response
cat_col = 'Category' if 'Category' in df.columns else df.columns[0]
acre_col = 'Acres' if 'Acres' in df.columns else [c for c in df.columns if 'acre' in c.lower() or 'area' in c.lower()]
acre_col = acre_col[0] if isinstance(acre_col, list) and acre_col else (acre_col or df.select_dtypes(include=[np.number]).columns[0])

top_crops = df.groupby(cat_col)[acre_col].sum().sort_values(ascending=False).head(15)
top_crops.plot(kind='barh', figsize=(10, 6))
plt.title('Top 15 Crop Categories by Total Acreage (Sample Counties)')
plt.xlabel('Total Acres')
plt.tight_layout()
plt.show()

In [ ]:
# Total acreage by year
year_totals = df.groupby('YEAR')[acre_col].sum()
plt.figure(figsize=(8, 4))
plt.bar(year_totals.index, year_totals.values)
plt.title('Total Crop Acreage by Year (Sample Counties)')
plt.xlabel('Year')
plt.ylabel('Acres')
plt.show()

In [ ]:
# Focus on one county: top crops over time
this_fips = SAMPLE_FIPS[0]
df_county = df[df['FIPS'] == this_fips].copy()
df_county_top = df_county.groupby([cat_col, 'YEAR'])[acre_col].sum().unstack(fill_value=0)
df_county_top['Total'] = df_county_top.sum(axis=1)
df_county_top = df_county_top.nlargest(10, 'Total').drop(columns=['Total'])
df_county_top.T.plot(kind='bar', stacked=False, figsize=(10, 5), legend=True, title=f'Top Crops by Year - FIPS {this_fips}')
plt.xticks(rotation=0)
plt.xlabel('Year')
plt.ylabel('Acres')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()